In [ ]:
import os
import re
from pathlib import Path
import pandas as pd

OUTROOT_FULL   = Path("<PATH_TO_CI_GWAS_OUT>")
OUTROOT_PRUNED = Path("<PATH_TO_CI_GWAS_OUT_PRUNED>")

ALPHA_EXP = 3
MAX_LEVEL = 3
DEPTH = 1
RUNID = f"e{ALPHA_EXP}_l{MAX_LEVEL}_d{DEPTH}"

SETUPS_POOLED = [
    "sbp_pre_post_1to60_no_cvd",
    "dbp_pre_post_1to60_no_cvd",
]
SETUPS_AGE5 = [
    "sbp_pre_post_1to60_age5_with_statins_no_cvd",
    "dbp_pre_post_1to60_age5_with_statins_no_cvd",
]
SELECTED_MARKERS_FILENAME = "<PATH_TO_CUSKSS_SELECTED_MARKERS_TSV>"

AGE5_LABELS = {
    0: "pooled",
    1: "< 50",
    2: "50 - 55",
    3: "56 - 60",
    4: "60 - 64",
    5: "65+",
}

OUT_TSV = Path("<PATH_TO_SNP_LOF_COUNTS_FULL_VS_PRUNED_TSV>")

def _read_selected_markers(outroot, setup):
    fpath = outroot / setup / RUNID / SELECTED_MARKERS_FILENAME
    if not fpath.exists():
        raise FileNotFoundError(f"Missing file: {fpath}")

    df = pd.read_csv(fpath, sep="\t")

    for col in ["phenotype", "rsID"]:
        if col not in df.columns:
            raise ValueError(f"{fpath} is missing required column '{col}'. Columns: {list(df.columns)}")

    df = df.copy()
    df["setup"] = setup
    df["outroot"] = str(outroot)
    return df

def load_all_selected_markers(outroot, setups, source_label):
    dfs = []
    missing = []
    for s in setups:
        try = _read_selected_markers(outroot, s)
            d["source"] = source_label
            dfs.append(d)
        except FileNotFoundError as e:
            missing.append(str(e))

    if missing:
        msg = "\n".join(missing)
        raise FileNotFoundError(f"Some expected inputs are missing under outroot={outroot}:\n{msg}")

    return pd.concat(dfs, ignore_index=True)

import re
import pandas as pd

AGE_LABELS = {
    "pooled": "pooled",
    1: "< 50",
    2: "50 - 55",
    3: "56 - 60",
    4: "61 - 64",
    5: "65+",
}

TRAIT_LABELS = {
    ("SBP", "pre"):  "SBP pre-treatment",
    ("SBP", "post"): "SBP post-treatment",
    ("DBP", "pre"):  "DBP pre-treatment",
    ("DBP", "post"): "DBP post-treatment",
}

def _annotate_trait_age(df):

    df = df.copy()

    ph = df["phenotype"].astype(str)
    m = ph.str.extract(r"^(SBP|DBP)_(pre|post)(?:_(\d+))?_ADJ$", expand=True)
    df["bp_trait"] = m[0]
    df["when"] = m[1]
    df["age_group_num"] = pd.to_numeric(m[2], errors="coerce")  # NaN => pooled

    df = df[df["bp_trait"].notna()].copy()

    df["age_group_num"] = df["age_group_num"].fillna("pooled")
    df["Age"] = df["age_group_num"].map(lambda x: AGE_LABELS.get(int(x), str(x)) if x != "pooled" else AGE_LABELS["pooled"])

    df["Trait"] = df.apply(lambda r: TRAIT_LABELS[(r["bp_trait"], r["when"])], axis=1)

    rid = df["rsID"].astype(str)
    is_rs = rid.str.match(r"^rs\d+$", case=False, na=False)
    is_chrpos = rid.str.match(r"^(chr)?\d+:\d+.*$", case=False, na=False)
    df["is_snp"] = is_rs | is_chrpos
    df["is_lof"] = ~df["is_snp"]

    return df

def _counts_by_trait_age(df):
    """
    Returns long table with columns:
      Trait, Age, n_snps, n_lof
    """
    df = df.drop_duplicates(["setup", "phenotype", "rsID"])  # robust against repeats
    out = (
        df.groupby(["Trait", "Age"], as_index=False)
          .agg(
              n_snps=("is_snp", "sum"),
              n_lof=("is_lof", "sum"),
          )
    )
    return out

def infer_trait(phenotype):
    p = str(phenotype)
    has_sbp = re.search(r"\bSBP\b", p, flags=re.I) is not None
    has_dbp = re.search(r"\bDBP\b", p, flags=re.I) is not None

    if has_sbp and re.search(r"post", p, flags=re.I):
        return "SBP post-treatment"
    if has_sbp and re.search(r"pre", p, flags=re.I):
        return "SBP pre-treatment"
    if has_dbp and re.search(r"post", p, flags=re.I):
        return "DBP post-treatment"
    if has_dbp and re.search(r"pre", p, flags=re.I):
        return "DBP pre-treatment"
    return None

def is_snp_marker(rsid):
    return str(rsid).lower().startswith("rs")

def infer_age_group_ix(df):

    for c in ["age_group_ix", "age_group_id", "age_group", "age", "age_bin"]:
        if c in df.columns:
            s = df[c]
            if s.dtype == "object":
                extracted = s.astype(str).str.extract(r"([1-5])", expand=False)
                out = pd.to_numeric(extracted, errors="coerce")
            else = pd.to_numeric(s, errors="coerce")
            return out.fillna(0).astype("int64")

    text = (df["phenotype"].astype(str) + " " + df["setup"].astype(str))
    extracted = text.str.extract(r"age5g([1-5])", expand=False)
    out = pd.to_numeric(extracted, errors="coerce")

    is_age_setup = df["setup"].astype(str).str.contains("age5", na=False)
    out = out.where(is_age_setup, 0)

    if out[is_age_setup].isna().any():
        bad = df.loc[is_age_setup & out.isna(), ["setup", "phenotype"]].drop_duplicates().head(20)
        raise ValueError(
            "Could not infer age5 group (expecting 'age5g1..age5g5' in phenotype or setup) "
            f"for some rows. Examples:\n{bad.to_string(index=False)}"
        )

    return out.fillna(0).astype("int64")

def summarize_counts(df):
    df = df.copy()
    df["trait"] = df["phenotype"].map(infer_trait)
    df = df[df["trait"].notna()].copy()

    df["age_group_ix"] = infer_age_group_ix(df)
    df["age"] = df["age_group_ix"].map(AGE5_LABELS)

    df["marker_class"] = df["rsID"].map(lambda x: "SNP" if is_snp_marker(x) else "LoF")

    g = (
        df.groupby(["trait", "age", "marker_class"])["rsID"]
          .nunique()
          .unstack(fill_value=0)
          .reset_index()
    )

    for c in ["SNP", "LoF"]:
        if c not in g.columns:
            g[c] = 0

    return g.rename(columns={"SNP": "n_snp", "LoF": "n_lof"})

setups = SETUPS_POOLED + SETUPS_AGE5

df_full = load_all_selected_markers(OUTROOT_FULL, setups, source_label="full")
df_pruned = load_all_selected_markers(OUTROOT_PRUNED, setups, source_label="ld_pruned")

df_full = df_full[df_full["p_fdr"]<0.05]

df_pruned = df_pruned[df_pruned["p_fdr"]<0.05]

def build_final_table(df_full, df_pruned):
    full = _counts_by_trait_age(_annotate_trait_age(df_full))
    prun = _counts_by_trait_age(_annotate_trait_age(df_pruned))

    tab = full.merge(
        prun,
        on=["Trait", "Age"],
        how="outer",
        suffixes=("_full", "_pruned"),
    ).fillna(0)

    for c in ["n_snps_full", "n_lof_full", "n_snps_pruned", "n_lof_pruned"]:
        tab[c] = tab[c].astype(int)

    tab = tab.rename(columns={
        "n_snps_full": "SNPs",
        "n_lof_full": "LoF",
        "n_snps_pruned": "SNPs_pruned",
        "n_lof_pruned": "LoF_pruned",
    })

    tab.columns = [
        ("Trait", "") if col == "Trait" else
        ("Age", "") if col == "Age" else
        ("all SNPs", "#SNPs") if col == "SNPs" else
        ("all SNPs", "#LoF") if col == "LoF" else
        ("LD pruned", "#SNPs") if col == "SNPs_pruned" else
        ("LD pruned", "#LoF") if col == "LoF_pruned" else
        col
        for col in tab.columns
    ]
    tab = tab.sort_values([("Trait",""), ("Age","")])

    trait_order = [
        "SBP pre-treatment",
        "SBP post-treatment",
        "DBP pre-treatment",
        "DBP post-treatment",
    ]
    age_order = ["pooled", "< 50", "50 - 55", "56 - 60", "60 - 64", "65+"]

    tab[("Trait","")] = pd.Categorical(tab[("Trait","")], categories=trait_order, ordered=True)
    tab[("Age","")] = pd.Categorical(tab[("Age","")], categories=age_order, ordered=True)
    tab = tab.sort_values([("Trait",""), ("Age","")]).reset_index(drop=True)

    return tab

table = build_final_table(df_full, df_pruned)

flat = table.copy()
flat.columns = [
    "Trait" if c2 == "Trait" else "Age" if c2 == "Age" else f"{c1} {c2}"
    for c1, c2 in flat.columns
]
flat.to_csv(OUT_TSV, sep="\t", index=False)
import numpy as np
import matplotlib.pyplot as plt

COLOR_LOF = "#9DB4A5"
COLOR_SNP = "#5B7D8D"

AGE_ORDER = ["pooled", "< 50", "50 - 55", "56 - 60", "60 - 64", "65+"]

ct = _counts_by_trait_age(_annotate_trait_age(df_full)).set_index(["Trait", "Age"])

def _get(trait, age, col):
    try:
        return int(ct.loc[(trait, age), col])
    except KeyError:
        return 0

from matplotlib.ticker import FixedLocator, FuncFormatter

def set_countlike_log_ticks(ax):
    y0, y1 = ax.get_ylim()
    desired_counts = [1]
    v = 10
    while (v + 1) <= y1:
        desired_counts.append(v)
        v *= 10

    ticks = [c + 1 for c in desired_counts if y0 <= (c + 1) <= y1]
    ax.yaxis.set_major_locator(FixedLocator(ticks))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _))}"))

def _panel(ax, bp):
    pre, post = f"{bp} pre-treatment", f"{bp} post-treatment"

    labels, lof, snp = [], [], []
    for age in AGE_ORDER:
        labels += [
            f"{bp}-pre all ages" if age == "pooled" else f"pre {age}",
            "post all ages"      if age == "pooled" else f"post {age}",
        ]
        lof += [_get(pre, age, "n_lof"),  _get(post, age, "n_lof")]
        snp += [_get(pre, age, "n_snps"), _get(post, age, "n_snps")]

    lof = np.asarray(lof, dtype=float)
    snp = np.asarray(snp, dtype=float)
    tot = lof + snp

    mask = tot > 0
    lof = np.where(mask, lof, np.nan)
    snp = np.where(mask, snp, np.nan)

    x = np.arange(len(labels))

    base = np.where(mask, 1.0, np.nan)

    ax.bar(x, lof, bottom=base, color=COLOR_LOF, label="LoF")
    ax.bar(x, snp, bottom=base + np.nan_to_num(lof, nan=0.0), color=COLOR_SNP, label="SNP")

    ax.set_yscale("log")
    ax.set_ylim(1, None)
    set_countlike_log_ticks(ax)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_title(f"{bp}")
    ax.set_ylabel("Number of variants")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
_panel(ax1, "SBP"); ax1.legend(loc="best")
_panel(ax2, "DBP")
fig.tight_layout()
fig.savefig("<PATH_TO_BP_STACKED_BARS_PNG>", dpi=150, bbox_inches="tight")
plt.show()

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

AGE_BINS = {"1":"<50", "2":"50Ă„â€šĂ‹ÂÄ‚ËĂ˘â‚¬ĹˇĂ‚Â¬Ä‚ËĂ˘â€šÂ¬Äąâ€ş55", "3":"56Ă„â€šĂ‹ÂÄ‚ËĂ˘â‚¬ĹˇĂ‚Â¬Ä‚ËĂ˘â€šÂ¬Äąâ€ş60", "4":"61Ă„â€šĂ‹ÂÄ‚ËĂ˘â‚¬ĹˇĂ‚Â¬Ä‚ËĂ˘â€šÂ¬Äąâ€ş64", "5":"65+"}

def _slice(df, setup): return df[df["setup"] == setup].copy()

def _variants(df, phenotype):
    d = df.loc[df["phenotype"] == phenotype, ["rsID"] + [c for c in ["chr","bp"] if c in df.columns]].drop_duplicates()
    d["rsID"] = d["rsID"].astype(str)

    if "chr" not in d.columns: d["chr"] = np.nan
    if "bp"  not in d.columns: d["bp"]  = np.nan

    m = d["rsID"].str.extract(r"^(?:chr)?(\d+):(\d+)", expand=True)
    d["chr"] = pd.to_numeric(d["chr"], errors="coerce").fillna(pd.to_numeric(m[0], errors="coerce"))
    d["bp"]  = pd.to_numeric(d["bp"],  errors="coerce").fillna(pd.to_numeric(m[1], errors="coerce"))
    return d

def _match_count(a, b, window_bp=50_000):
    # exact matches (SNP or LoF)
    exact = set(a["rsID"]).intersection(set(b["rsID"]))
    k_exact = len(exact)
    a = a[~a["rsID"].isin(exact)].copy()
    b = b[~b["rsID"].isin(exact)].copy()

    # proximity matches for rows with positions
    a = a.dropna(subset=["chr","bp"])
    b = b.dropna(subset=["chr","bp"])
    if a.empty or b.empty:
        return k_exact

    k_near = 0
    for chrom in np.intersect1d(a["chr"].unique(), b["chr"].unique()):
        aa = np.sort(a.loc[a["chr"] == chrom, "bp"].to_numpy(dtype=np.int64))
        bb = np.sort(b.loc[b["chr"] == chrom, "bp"].to_numpy(dtype=np.int64))
        i = j = 0
        while i < len(aa) and j < len(bb):
            if bb[j] < aa[i] - window_bp:
                j += 1
            elif bb[j] > aa[i] + window_bp:
                i += 1
            else = 1
                i += 1
                j += 1

    return k_exact + k_near

def _venn(ax, A_df, B_df, labelA, labelB, color_all="#9DB4A5", color_bin="#5B7D8D", alpha=0.7, window_bp=10_000):
    nA = A_df["rsID"].nunique()
    nB = B_df["rsID"].nunique()
    k  = _match_count(A_df, B_df, window_bp=window_bp)
    k = int(min(k, nA, nB))

    if (nA + nB + k) == 0:
        ax.axis("off")
        ax.text(0.5, 0.5, "no variants", ha="center", va="center")
        return

    venn2(
        subsets=(nA - k, nB - k, k),
        set_labels=(labelA, labelB),
        set_colors=(color_all, color_bin),
        alpha=alpha,
        ax=ax,
    )

def four_venn_plots_from_combined_50kb(
    combined_df,
    output_png="<PATH_TO_FOUR_VENNS_50KB_PNG>",
    setup_all_sbp="sbp_pre_post_1to60_no_cvd",
    setup_age5_sbp="sbp_pre_post_1to60_age5_with_statins_no_cvd",
    setup_all_dbp="dbp_pre_post_1to60_no_cvd",
    setup_age5_dbp="dbp_pre_post_1to60_age5_with_statins_no_cvd",
    window_bp=50_000,
    color_all="#9DB4A5",
    color_bin="#5B7D8D",
    alpha=0.7,
    font_size=18,
):
    plt.rcParams.update({
        "font.size": font_size,
        "axes.titlesize": font_size,
        "axes.labelsize": font_size,
        "xtick.labelsize": font_size - 2,
        "ytick.labelsize": font_size - 2,
    })

    DBP_all = _slice(combined_df, setup_all_dbp)
    DBP_5   = _slice(combined_df, setup_age5_dbp)
    SBP_all = _slice(combined_df, setup_all_sbp)
    SBP_5   = _slice(combined_df, setup_age5_sbp)

    fig, axes = plt.subplots(1, 4, figsize=(22, 8))
    axes = axes.ravel()

    _venn(axes[0], _variants(DBP_all,"DBP_pre_ADJ"), _variants(DBP_5,"DBP_pre_1_ADJ"),
          "DBP: all ages", f"DBP: {AGE_BINS['1']}", color_all, color_bin, alpha, window_bp)
    _venn(axes[1], _variants(DBP_all,"DBP_pre_ADJ"), _variants(DBP_5,"DBP_pre_2_ADJ"),
          "DBP: all ages", f"DBP: {AGE_BINS['2']}", color_all, color_bin, alpha, window_bp)

    _venn(axes[2], _variants(SBP_all,"SBP_pre_ADJ"), _variants(SBP_5,"SBP_pre_1_ADJ"),
          "SBP: all ages", f"SBP: {AGE_BINS['1']}", color_all, color_bin, alpha, window_bp)
    _venn(axes[3], _variants(SBP_all,"SBP_pre_ADJ"), _variants(SBP_5,"SBP_pre_2_ADJ"),
          "SBP: all ages", f"SBP: {AGE_BINS['2']}", color_all, color_bin, alpha, window_bp)

    fig.tight_layout()
    fig.savefig(output_png, dpi=150, bbox_inches="tight")
    fig.savefig("<PATH_TO_VENNS_FIGURE_3_B_50KB_SVG>", format="svg", bbox_inches="tight")
    plt.show()

four_venn_plots_from_combined_50kb(df_full, window_bp=50_000, font_size=16)

df_full.to_csv("<PATH_TO_MAIN_SETUP_RESULTS_TSV>", sep="\t")